# Import Libraries

In [2]:
import pandas as pd
import requests
import time

# Load Required Original Dataset 

In [ ]:
# ---------------------------------------------------------
# 1. Load the Kaggle AQI data using Data Loader
# ---------------------------------------------------------

from src.data_loader import load_station_hour, load_stations

df_station_hour = load_station_hour()
df_stations = load_stations()

# Preprocessing

In [ ]:
# Convert timestamps
df_station_hour["Datetime"] = pd.to_datetime(df_station_hour["Datetime"])

# ---------------------------------------------------------
# 2. Connect each StationId to its City
# ---------------------------------------------------------

df_station_city = df_stations[["StationId", "City"]].drop_duplicates()

df_station_hour = df_station_hour.merge(
    df_station_city,
    on="StationId",
    how="left"
)

# Check the actual cities present in our hourly AQI data
df_cities = sorted(df_station_hour["City"].dropna().unique())

print("Number of cities:", len(df_cities))
print("\nCities:")
print(df_cities)

# ---------------------------------------------------------
# 3. Coordinates for the 26 cities
# ---------------------------------------------------------

locations = {
    "Amaravati": (16.5062, 80.6480),
    "Visakhapatnam": (17.6868, 83.2185),
    "Guwahati": (26.1445, 91.7362),
    "Patna": (25.5941, 85.1376),
    "Chandigarh": (30.7333, 76.7794),
    "Delhi": (28.6139, 77.2090),
    "Ahmedabad": (23.0225, 72.5714),
    "Gurugram": (28.4595, 77.0266),
    "Jorapokhar": (23.7380, 86.4170),
    "Bengaluru": (12.9716, 77.5946),
    "Ernakulam": (9.9816, 76.2999),
    "Kochi": (9.9312, 76.2673),
    "Thiruvananthapuram": (8.5241, 76.9366),
    "Mumbai": (19.0760, 72.8777),
    "Shillong": (25.5788, 91.8933),
    "Bhopal": (23.2599, 77.4126),
    "Aizawl": (23.7271, 92.7176),
    "Brajrajnagar": (21.8159, 83.9213),
    "Talcher": (20.9538, 85.2285),
    "Amritsar": (31.6340, 74.8723),
    "Jaipur": (26.9124, 75.7873),
    "Hyderabad": (17.3850, 78.4867),
    "Chennai": (13.0827, 80.2707),
    "Coimbatore": (11.0168, 76.9558),
    "Lucknow": (26.8467, 80.9462),
    "Kolkata": (22.5726, 88.3639)
}

# Check whether every AQI city has coordinates
missing_coordinates = [
    city for city in df_cities
    if city not in locations
]

print("\nCities without coordinates:", missing_coordinates)

# Extract Data from Open-Meteo

In [3]:
# ---------------------------------------------------------
# 4. Extract Open-Meteo weather data
#    for each city's actual AQI observation period
# ---------------------------------------------------------

all_weather = []

for city in df_cities:

    latitude, longitude = locations[city]

    # Get this city's actual AQI observation period
    df_city_data = df_station_hour[
        df_station_hour["City"] == city
    ]

    start_date = df_city_data["Datetime"].min().strftime("%Y-%m-%d")
    end_date = df_city_data["Datetime"].max().strftime("%Y-%m-%d")

    print(
        f"\nFetching {city}: "
        f"{start_date} → {end_date}"
    )

    url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={latitude}&"
        f"longitude={longitude}&"
        f"start_date={start_date}&"
        f"end_date={end_date}&"
        "hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&"
        "timezone=Asia/Kolkata"
    )

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    data = response.json()

    if "hourly" not in data:
        print(f"No hourly data returned for {city}")
        continue

    weather_df = pd.DataFrame({
        "City": city,
        "Datetime": data["hourly"]["time"],
        "Temperature_C": data["hourly"]["temperature_2m"],
        "Relative_Humidity_pct": data["hourly"]["relative_humidity_2m"],
        "Wind_Speed_kmh": data["hourly"]["wind_speed_10m"]
    })

    weather_df["Datetime"] = pd.to_datetime(
        weather_df["Datetime"]
    )

    all_weather.append(weather_df)

    # Small pause between API requests
    time.sleep(1)

# ---------------------------------------------------------


/tmp/ipykernel_1090/1813189372.py:5: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  station_hour = pd.read_csv("/content/drive/MyDrive/AQI_Prediction_Kanchana/Kaggle_India_AQI_Data/station_hour.csv")


Number of cities: 26

Cities:
['Ahmedabad', 'Aizawl', 'Amaravati', 'Amritsar', 'Bengaluru', 'Bhopal', 'Brajrajnagar', 'Chandigarh', 'Chennai', 'Coimbatore', 'Delhi', 'Ernakulam', 'Gurugram', 'Guwahati', 'Hyderabad', 'Jaipur', 'Jorapokhar', 'Kochi', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna', 'Shillong', 'Talcher', 'Thiruvananthapuram', 'Visakhapatnam']

Cities without coordinates: []

Fetching Ahmedabad: 2015-01-01 → 2020-07-01

Fetching Aizawl: 2020-03-11 → 2020-07-01

Fetching Amaravati: 2017-11-24 → 2020-07-01

Fetching Amritsar: 2017-02-27 → 2020-07-01

Fetching Bengaluru: 2015-01-01 → 2020-07-01

Fetching Bhopal: 2019-09-17 → 2020-07-01

Fetching Brajrajnagar: 2017-12-07 → 2020-07-01

Fetching Chandigarh: 2019-09-02 → 2020-07-01

Fetching Chennai: 2015-01-01 → 2020-07-01

Fetching Coimbatore: 2019-06-12 → 2020-07-01

Fetching Delhi: 2015-01-01 → 2020-07-01

Fetching Ernakulam: 2020-01-22 → 2020-07-01

Fetching Gurugram: 2015-11-27 → 2020-07-01

Fetching Guwahati: 2019-02-16 → 2020-07

In [ ]:
# 5. Combine all cities
# ---------------------------------------------------------

weather_data = pd.concat(
    all_weather,
    ignore_index=True
)

# ---------------------------------------------------------
# 6. Save the corrected weather dataset
# ---------------------------------------------------------

output_file = "/content/openmeteo_india_weather.csv"

weather_data.to_csv(
    output_file,
    index=False
)

print("\n---------------------------------------")
print("Weather extraction completed")
print("---------------------------------------")

# Weather Data EDA

In [6]:
print("Shape:", weather_data.shape)

print("\nDate range:")
print(weather_data["Datetime"].min())
print(weather_data["Datetime"].max())

print("\nCities:", weather_data["City"].nunique())

print("\nMissing values:")
print(weather_data.isna().sum())


---------------------------------------
Weather extraction completed
---------------------------------------
Shape: (708744, 5)

Date range:
2015-01-01 00:00:00
2020-07-01 23:00:00

Cities: 26

Missing values:
City                     0
Datetime                 0
Temperature_C            0
Relative_Humidity_pct    0
Wind_Speed_kmh           0
dtype: int64


In [7]:
weather_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 708744 entries, 0 to 708743
Data columns (total 5 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   City                   708744 non-null  object        
 1   Datetime               708744 non-null  datetime64[ns]
 2   Temperature_C          708744 non-null  float64       
 3   Relative_Humidity_pct  708744 non-null  int64         
 4   Wind_Speed_kmh         708744 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 27.0+ MB


In [8]:
# Check the extracted weather dataset

print("First 5 rows:")
print(weather_data.head())

print("\nLast 5 rows:")
print(weather_data.tail())

print("\nData types:")
print(weather_data.dtypes)

print("\nRows per city:")
print(weather_data["City"].value_counts().sort_index())

First 5 rows:
        City            Datetime  Temperature_C  Relative_Humidity_pct  \
0  Ahmedabad 2015-01-01 00:00:00           18.4                     64   
1  Ahmedabad 2015-01-01 01:00:00           18.1                     69   
2  Ahmedabad 2015-01-01 02:00:00           17.8                     73   
3  Ahmedabad 2015-01-01 03:00:00           17.7                     74   
4  Ahmedabad 2015-01-01 04:00:00           17.4                     77   

   Wind_Speed_kmh  
0            12.8  
1            14.1  
2            14.4  
3            13.6  
4            13.3  

Last 5 rows:
                 City            Datetime  Temperature_C  \
708739  Visakhapatnam 2020-07-01 19:00:00           28.3   
708740  Visakhapatnam 2020-07-01 20:00:00           28.1   
708741  Visakhapatnam 2020-07-01 21:00:00           28.0   
708742  Visakhapatnam 2020-07-01 22:00:00           27.7   
708743  Visakhapatnam 2020-07-01 23:00:00           27.5   

        Relative_Humidity_pct  Wind_Speed_kmh 

In [9]:
# Check how many AQI observations have matching weather data

aqi_weather_check = df_station_hour.merge(
    weather_data[[
        "City",
        "Datetime",
        "Temperature_C",
        "Relative_Humidity_pct",
        "Wind_Speed_kmh"
    ]],
    on=["City", "Datetime"],
    how="left",
    indicator=True
)

print("Total AQI rows:", len(aqi_weather_check))

print("\nWeather match status:")
print(aqi_weather_check["_merge"].value_counts())

print("\nWeather missing percentage:")
missing_weather = (
    aqi_weather_check["_merge"].eq("left_only").mean() * 100
)

print(f"{missing_weather:.2f}%")

Total AQI rows: 2589083

Weather match status:
_merge
both          2589083
left_only           0
right_only          0
Name: count, dtype: int64

Weather missing percentage:
0.00%


In [10]:
aqi_weather_check.head()

,StationId,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,Benzene,Toluene,Xylene,AQI,AQI_Bucket,City,Temperature_C,Relative_Humidity_pct,Wind_Speed_kmh,_merge
0,AP001,2017-11-24 17:00:00,60.50,98.00,2.35,30.80,18.25,8.50,0.1,11.85,...,0.1,6.10,0.10,NaN,NaN,Amaravati,31.7,46,3.5,both
1,AP001,2017-11-24 18:00:00,65.50,111.25,2.70,24.20,15.07,9.77,0.1,13.17,...,0.1,6.25,0.15,NaN,NaN,Amaravati,28.8,63,13.4,both
2,AP001,2017-11-24 19:00:00,80.00,132.00,2.10,25.18,15.15,12.02,0.1,12.08,...,0.2,5.98,0.18,NaN,NaN,Amaravati,26.7,78,10.5,both
3,AP001,2017-11-24 20:00:00,81.50,133.25,1.95,16.25,10.23,11.58,0.1,10.47,...,0.2,6.72,0.10,NaN,NaN,Amaravati,26.2,81,9.5,both
4,AP001,2017-11-24 21:00:00,75.25,116.00,1.43,17.48,10.43,12.03,0.1,9.12,...,0.2,5.75,0.08,NaN,NaN,Amaravati,25.6,84,7.1,both


In [14]:
df_city_data.shape

(35053, 17)

In [11]:
df_station_hour.shape

(2589083, 17)

In [12]:
weather_data.shape

(708744, 5)

In [13]:
aqi_weather_check.shape

(2589083, 21)

We have now established:  

* Kaggle AQI data → 2,589,083 rows  
* Open-Meteo weather → 26 cities
* Match using City + Datetime → 100% matched
* Weather missing after matching → 0%